In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%%RecordEventWithColumnInfo
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
import pandas as pd
# from tpch import load_lineitem, load_orders, load_customer, load_nation, q10
DATA_ROOT = "/home/dias-benchmarks/tpch/data"
STORAGE_OPTS = {}



In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)



In [ ]:
%%RecordEventWithColumnInfo
### cell 2 ###


def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%%RecordEventWithColumnInfo
### cell 3 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 4 ###


date1 = pd.Timestamp("1994-11-01")
date2 = pd.Timestamp("1995-02-01")
osel = (orders.O_ORDERDATE >= date1) & (orders.O_ORDERDATE < date2)
lsel = lineitem.L_RETURNFLAG == "R"
forders = orders[osel]
flineitem = lineitem[lsel]
jn1 = flineitem.merge(forders, left_on="L_ORDERKEY", right_on="O_ORDERKEY")
jn2 = jn1.merge(customer, left_on="O_CUSTKEY", right_on="C_CUSTKEY")
jn3 = jn2.merge(nation, left_on="C_NATIONKEY", right_on="N_NATIONKEY")
jn3["TMP"] = jn3.L_EXTENDEDPRICE * (1.0 - jn3.L_DISCOUNT)
gb = jn3.groupby(
    [
        "C_CUSTKEY",
        "C_NAME",
        "C_ACCTBAL",
        "C_PHONE",
        "N_NAME",
        "C_ADDRESS",
        "C_COMMENT",
    ],
    as_index=False,
    sort=False,
)["TMP"].sum()
total = gb.sort_values("TMP", ascending=False)



In [ ]:
%%RecordEventWithColumnInfo
### cell 5 ###

total.info()